# D2：在 Notebook 中使用 seekdb

当前 SDK 使用 `pyseekdb.Client` 连接数据库，再通过 `Collection` 写入和检索。
本 Notebook 复用课程的连接工厂与 `code/.env`，保留原来的 5 条示例文档。

请使用项目 `.venv` 内核并从上到下运行。数据库服务需已启动；本实验使用本地向量模型，无需调用 DeepSeek 聊天接口。

## 1. 连接数据库并写入文档

`get_or_create_collection()` 获取或创建集合，`upsert()` 按 ID 写入或更新。
正文用于向量检索，`category`、`version` 保存为可用于过滤的元数据。

In [3]:
import sys
from pathlib import Path

# Notebook 的工作目录可能是项目根目录或 code/D2，向上寻找公共配置模块。
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "code" / "config.py").is_file():
        code_dir = parent / "code"
        break
    if (parent / "config.py").is_file() and (parent / "seekdb_runtime.py").is_file():
        code_dir = parent
        break
else:
    raise RuntimeError("请从 easy-data-x-ai 项目或 code/D2 目录打开 Notebook")

if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

import pyseekdb
from seekdb_runtime import create_seekdb_client, require_destructive_seekdb_access
from D2.db_lifecycle import close_database
from D2.d2_3_hybrid_search import print_results

print("当前 Python：", sys.executable)
print("pyseekdb 版本：", pyseekdb.__version__)

# 公共客户端工厂自动读取 code/.env 中的 SEEKDB_* 配置。
# 使用独立的 Notebook 集合，避免影响 d2_1 创建的课程知识库。
COLLECTION_NAME = "d2_notebook_knowledge_base"
require_destructive_seekdb_access("写入 D2 Notebook 演示集合")

# 重跑本单元格前释放上次连接；关闭连接不会删除集合。
previous_db = globals().get("db")
if previous_db is not None:
    close_database(previous_db)
db = None
print("正在连接 seekdb 并加载向量模型（首次可能需要下载）...", flush=True)
db = create_seekdb_client(path=code_dir / "D2" / "seekdb")
collection = db.get_or_create_collection(name=COLLECTION_NAME)

# 保留原来的示例文档；正文与结构化元数据分别传入 SDK。
docs = [
    {
        "content": "错误码 E-4012 表示数据库连接超时。解决方案：检查网络配置，确认数据库服务端口是否开放，建议超时时间设置为 30 秒。",
        "category": "error_codes",
        "version": "4.2"
    },
    {
        "content": "错误码 E-4013 表示认证失败。解决方案：检查用户名和密码是否正确，确认账户是否被锁定。",
        "category": "error_codes",
        "version": "4.2"
    },
    {
        "content": "数据库查询性能优化指南：合理使用索引可以将查询速度提升 10 倍以上。建议对高频查询的 WHERE 条件列建立索引。",
        "category": "best_practices",
        "version": "4.2"
    },
    {
        "content": "访问控制架构设计：基于 RBAC 模型实现用户权限管理，支持角色继承和细粒度的资源级权限控制。",
        "category": "architecture",
        "version": "4.1"
    },
    {
        "content": "OB-4.2.1 版本新特性：支持在线 DDL 操作、改进了并行查询引擎、修复了分区表在特定条件下的数据倾斜问题。",
        "category": "release_notes",
        "version": "4.2.1"
    }
]


# ids、documents、metadatas 按相同下标对应。
# 不传 embeddings 时，SDK 使用默认本地向量模型生成文档向量。
# upsert 对相同 ID 执行更新，让单元格可以重复运行而不重复插入。
collection.upsert(
    ids=[f"notebook_doc_{i}" for i in range(len(docs))],
    documents=[doc["content"] for doc in docs],
    metadatas=[{"category": doc["category"], "version": doc["version"]} for doc in docs],
)
print(f"集合 {COLLECTION_NAME} 已就绪，共 {collection.count()} 条文档")

当前 Python： d:\aaaMyProgram\easy-data-x-ai\.venv\Scripts\python.exe
pyseekdb 版本： 1.4.0.post1
正在连接 seekdb 并加载向量模型（首次可能需要下载）...


d:\aaaMyProgram\easy-data-x-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 31633.10it/s]


集合 d2_notebook_knowledge_base 已就绪，共 5 条文档


## 2. 精确编号的混合检索

`query` 支路按正文词项寻找候选，`knn` 支路按问题向量寻找候选，`rank` 使用 RRF 融合名次。
各支路最多取 5 条，融合后最多返回 3 条。仅在 `query` 支路指定编号，不会把该编号变成整体硬过滤条件。

In [4]:
# hybrid_search 属于集合对象；当前 SDK 用 n_results 控制结果数量。
results = collection.hybrid_search(
    query={"where_document": {"$contains": "E-4012"}, "n_results": 5},
    knn={"query_texts": ["错误码 E-4012 的解决方案"], "n_results": 5},
    rank={"rrf": {}},
    n_results=3,
)
# SDK 结果按“查询批次 -> 命中列表”组织，复用课程函数展示文本、分数和元数据。
print_results(results, "混合检索：错误码 E-4012")

  混合检索：错误码 E-4012
  [1] 分数：0.0328  内容：错误码 E-4012 表示数据库连接超时。解决方案：检查网络配置，确认数据库服务端口是否开放，建议超时时间设置为 30 秒。
       分类：error_codes  版本：4.2
  [2] 分数：0.0161  内容：错误码 E-4013 表示认证失败。解决方案：检查用户名和密码是否正确，确认账户是否被锁定。
       分类：error_codes  版本：4.2
  [3] 分数：0.0159  内容：OB-4.2.1 版本新特性：支持在线 DDL 操作、改进了并行查询引擎、修复了分区表在特定条件下的数据倾斜问题。
       分类：release_notes  版本：4.2.1


## 3. 用户措辞与文档不同的查询

观察“用户权限”能否检索到“访问控制/RBAC”。模型与数据会影响排名，实际命中情况需要结合输出判断。

In [5]:
results = collection.hybrid_search(
    query={"where_document": {"$contains": "权限"}, "n_results": 5},
    knn={"query_texts": ["怎么设计用户权限"], "n_results": 5},
    rank={"rrf": {}},
    n_results=3,
)
print_results(results, "混合检索：用户权限")

  混合检索：用户权限
  [1] 分数：0.0320  内容：访问控制架构设计：基于 RBAC 模型实现用户权限管理，支持角色继承和细粒度的资源级权限控制。
       分类：architecture  版本：4.1
  [2] 分数：0.0164  内容：数据库查询性能优化指南：合理使用索引可以将查询速度提升 10 倍以上。建议对高频查询的 WHERE 条件列建立索引。
       分类：best_practices  版本：4.2
  [3] 分数：0.0161  内容：错误码 E-4012 表示数据库连接超时。解决方案：检查网络配置，确认数据库服务端口是否开放，建议超时时间设置为 30 秒。
       分类：error_codes  版本：4.2


## 4. 同时加入版本过滤

`where` 过滤结构化元数据，`where_document` 过滤正文内容。
要让所有候选都满足 `version=4.2`，必须给文本和向量两个支路都加上版本条件。

In [6]:
results = collection.hybrid_search(
    query={
        "where_document": {"$contains": "性能优化"},
        "where": {"version": "4.2"},
        "n_results": 5,
    },
    knn={
        "query_texts": ["性能优化"],
        "where": {"version": "4.2"},
        "n_results": 5,
    },
    rank={"rrf": {}},
    n_results=3,
)
print_results(results, "混合检索：性能优化，仅限 4.2 版本")

  混合检索：性能优化，仅限 4.2 版本
  [1] 分数：0.0328  内容：数据库查询性能优化指南：合理使用索引可以将查询速度提升 10 倍以上。建议对高频查询的 WHERE 条件列建立索引。
       分类：best_practices  版本：4.2
  [2] 分数：0.0161  内容：错误码 E-4012 表示数据库连接超时。解决方案：检查网络配置，确认数据库服务端口是否开放，建议超时时间设置为 30 秒。
       分类：error_codes  版本：4.2
  [3] 分数：0.0159  内容：错误码 E-4013 表示认证失败。解决方案：检查用户名和密码是否正确，确认账户是否被锁定。
       分类：error_codes  版本：4.2


## 5. 关闭连接

完成实验后运行下面的单元格。集合会保留，下次从第一个代码单元格运行即可重新连接。

In [7]:
close_database(db)
db = None
print("数据库连接已关闭，演示集合已保留。")

数据库连接已关闭，演示集合已保留。
